# 03 — Tier-1 MCTS-HNM Benchmarks: Multi-Seed Baselines & Hierarchy Ablations

**Feature**: `F4` (`04-tier1-benchmark-suite` | `EPIC-04`)  
**Architecture Decisions**: `AD-001` (Tier-1 Embedding Prototyping), `AD-002` (Semantic vs. Ward Confusion Tree), `AD-004` (Scenario 1 & 2 Regimes), `AD-005` (Strict GiantHunter vs. Enhanced MCTS-HNM), `AD-008` (Remote JupyterHub Execution)

This notebook orchestrates the core experimental axes across $N = 5$ random seeds:
1. **Axis 1 (Negative Samplers Comparison)**: Uniform Sampling vs. OHEM vs. Strict GiantHunter vs. Enhanced MCTS-HNM in Scenario 1 (One-vs-Rest, $\rho = 1:50$).
2. **Axis 2 (Hierarchy Geometry Ablation)**: Native Semantic Tree vs. Data-Driven Ward Confusion Tree under Enhanced MCTS-HNM.
3. **Axis 3 (Hyperparameter Sensitivity)**: Exploration coefficient $c \in \{0.5, 1.0, 2.0, 4.0\}$ and momentum $\alpha \in \{0.1, 0.3, 0.5, 0.7\}$.
4. **Axis 4 (Scenario 2 Curriculum)**: 100-way Multi-Class evaluation.
5. **Statistical Significance**: Paired Student's $t$-test and Wilcoxon signed-rank tests ($p$-values) and export to `results/`.

In [1]:
from pathlib import Path
import os
import sys

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from src.utils.experiment_config import (
    ExecutionMode,
    ExperimentConfig,
    get_isolated_paths,
    print_execution_banner,
)
from src.data.cifar100_taxonomy import get_cifar100_taxonomy
from src.data.extract_embeddings import load_embedding_cache
from src.training.benchmark_runner import (
    audit_scenario1_imbalance,
    run_benchmark,
    run_sensitivity_ablation,
)
from src.utils.seed import set_global_seed

# Execution mode: REAL by default for scientific runs; SMOKE for testing
EXECUTION_MODE = ExecutionMode.REAL
if os.environ.get("EXECUTION_MODE"):
    EXECUTION_MODE = ExecutionMode(os.environ["EXECUTION_MODE"].lower())
elif "pytest" in sys.modules:
    EXECUTION_MODE = ExecutionMode.SMOKE

config = ExperimentConfig(execution_mode=EXECUTION_MODE)
print_execution_banner(config, notebook_name="03_tier1_mcts_benchmarks")
paths = get_isolated_paths(config, repo_root=REPO_ROOT)

SEED = set_global_seed(42)
RESULTS_DIR = paths.results_dir
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Repository root: {REPO_ROOT} | Results directory: {RESULTS_DIR}")


Repository root: /home/mlima/workspace/mcts-hnm | Results directory: /home/mlima/workspace/mcts-hnm/results


## 1. Load Precomputed ResNet-18 Embeddings

Loads `embeddings/cifar100_resnet18.pt` generated by Notebook 01, or synthesizes a deterministic fixture if cache is absent.

In [2]:
CACHE_PATH = paths.embeddings_dir / "cifar100_resnet18.pt"

if not CACHE_PATH.is_file():
    if config.execution_mode == ExecutionMode.REAL:
        raise FileNotFoundError(
            f"Real CIFAR-100 embeddings not found at {CACHE_PATH}. Run Notebook 01 in REAL mode first."
        )
    else:
        from src.data.extract_embeddings import extract_or_generate_embeddings
        extract_or_generate_embeddings(
            output_path=CACHE_PATH,
            config=config,
            smoke_samples_per_class=30,
            smoke_feature_dim=64,
        )

bundle = load_embedding_cache(CACHE_PATH)
train_features = bundle["train"]["embeddings"]
train_labels = bundle["train"]["fine_labels"]
val_features = bundle["val"]["embeddings"]
val_labels = bundle["val"]["fine_labels"]
test_features = bundle["test"]["embeddings"]
test_labels = bundle["test"]["fine_labels"]
print(f"Loaded {config.execution_mode.value.upper()} embeddings: Train={tuple(train_features.shape)}, Val={tuple(val_features.shape)}, Test={tuple(test_features.shape)}")


Loaded cached embeddings: Train=(45000, 512), Val=(5000, 512), Test=(10000, 512)


## 2. Axis 1: Sampler Comparison in Scenario 1 (Uniform vs. OHEM vs. Strict vs. Enhanced MCTS)

Compares the four negative sampling strategies across multiple seeds on Scenario 1 (One-vs-Rest, target class `0` / `beaver`, imbalanced positive pool).

Evaluates the canonical imbalance ratio $\rho = 1:50$ by default, subsampling positive instances via `src.data.splits.build_scenario1_split()`. For full publication deliverables, `imbalance_ratios` can be expanded to `("1:10", "1:50", "1:100")`.

In [3]:
SEEDS = (42, 43, 44, 45, 46) if config.execution_mode == ExecutionMode.REAL else (42,)
num_steps = 100 if config.execution_mode == ExecutionMode.REAL else 10
eval_every = 25 if config.execution_mode == ExecutionMode.REAL else 5
IMBALANCE_RATIOS_AXIS1 = ("1:50",)  # Evaluated canonical ratio; can be expanded to ("1:10", "1:50", "1:100") for full grid

res_axis1 = run_benchmark(
    scenario="scenario1",
    train_features=train_features,
    train_labels=train_labels,
    test_features=test_features,
    test_labels=test_labels,
    strategies=("uniform", "ohem", "mcts_strict", "mcts_enhanced"),
    tree_types=("semantic",),
    seeds=SEEDS,
    output_dir=RESULTS_DIR,
    benchmark_name="tier1_axis1_samplers",
    num_steps=num_steps,
    eval_every=eval_every,
    target_class_scenario1=0,
    val_features=val_features,
    val_labels=val_labels,
    config=config,
    imbalance_ratios=IMBALANCE_RATIOS_AXIS1,
)

df_axis1 = pd.read_csv(RESULTS_DIR / "tier1_axis1_samplers.csv")
summary_axis1 = df_axis1.groupby("strategy")[["auprc", "f1", "fpr_at_95_recall"]].agg(["mean", "std"])
print("=== Axis 1: Sampler Comparison Results (Scenario 1) ===")
print(summary_axis1)
if res_axis1.get("statistical_comparisons"):
    print("\nStatistical tests against Enhanced MCTS-HNM:")
    for comp, stats in res_axis1["statistical_comparisons"].items():
        print(f"  {comp}: diff={stats['mean_diff']:.4f}, t_p={stats['t_pvalue']:.4e}, w_p={stats['wilcoxon_pvalue']:.4e}")


=== Axis 1: Sampler Comparison Results (Scenario 1) ===
                  auprc                  f1           fpr_at_95_recall  \
                   mean       std      mean       std             mean   
strategy                                                                 
mcts_enhanced  0.825468  0.005078  0.738781  0.027810         0.021354   
mcts_strict    0.809869  0.027870  0.494886  0.101601         0.014667   
ohem           0.857446  0.007532  0.720110  0.043613         0.012424   
uniform        0.819304  0.018044  0.572725  0.038613         0.015515   

                         
                    std  
strategy                 
mcts_enhanced  0.002800  
mcts_strict    0.001615  
ohem           0.001303  
uniform        0.001238  

Statistical tests against Enhanced MCTS-HNM:
  mcts_enhanced_semantic_vs_uniform_none: diff=0.0062, t_p=4.8272e-01, w_p=6.2500e-01
  mcts_enhanced_semantic_vs_ohem_none: diff=-0.0320, t_p=1.1603e-04, w_p=6.2500e-02
  mcts_enhanced_semantic_vs

## 3. Axis 2: Hierarchy Geometry Ablation (Semantic vs. Ward Confusion Tree)

Tests whether empirical visual error clustering (Ward link on validation confusion matrix) improves MCTS mining efficiency over static taxonomy.

In [4]:
res_axis2 = run_benchmark(
    scenario="scenario1",
    train_features=train_features,
    train_labels=train_labels,
    test_features=test_features,
    test_labels=test_labels,
    strategies=("mcts_enhanced",),
    tree_types=("semantic", "confusion_ward"),
    seeds=SEEDS,
    output_dir=RESULTS_DIR,
    benchmark_name="tier1_axis2_tree_ablation",
    num_steps=num_steps,
    eval_every=eval_every,
    target_class_scenario1=0,
    val_features=val_features,
    val_labels=val_labels,
    config=config,
)

df_axis2 = pd.read_csv(RESULTS_DIR / "tier1_axis2_tree_ablation.csv")
summary_axis2 = df_axis2.groupby("tree_type")[["auprc", "f1", "fpr_at_95_recall"]].agg(["mean", "std"])
print("=== Axis 2: Tree Geometry Ablation Results ===")
print(summary_axis2)


=== Axis 2: Tree Geometry Ablation Results ===
                   auprc                  f1           fpr_at_95_recall  \
                    mean       std      mean       std             mean   
tree_type                                                                 
confusion_ward  0.773452  0.021194  0.506264  0.026718         0.016889   
semantic        0.825468  0.005078  0.738781  0.027810         0.021354   

                        
                   std  
tree_type               
confusion_ward  0.0013  
semantic        0.0028  


## 4. Axis 3: Scenario 2 Multi-Class Curriculum Benchmark

Evaluates 100-way multi-class classification accuracy, macro-F1, and worst-10-class recall under MCTS curriculum.

In [5]:
s2_seeds = (42, 43) if config.execution_mode == ExecutionMode.REAL else (42,)
s2_steps = 60 if config.execution_mode == ExecutionMode.REAL else 10
s2_eval = 20 if config.execution_mode == ExecutionMode.REAL else 5

res_s2 = run_benchmark(
    scenario="scenario2",
    train_features=train_features,
    train_labels=train_labels,
    test_features=test_features,
    test_labels=test_labels,
    strategies=("uniform", "mcts_enhanced"),
    tree_types=("semantic",),
    seeds=s2_seeds,
    output_dir=RESULTS_DIR,
    benchmark_name="tier1_scenario2_curriculum",
    num_steps=s2_steps,
    eval_every=s2_eval,
    val_features=val_features,
    val_labels=val_labels,
    config=config,
)

df_s2 = pd.read_csv(RESULTS_DIR / "tier1_scenario2_curriculum.csv")
summary_s2 = df_s2.groupby("strategy")[["top1_accuracy", "macro_f1", "worst_10_class_recall"]].agg(["mean", "std"])
print("=== Scenario 2: Multi-Class Curriculum Results ===")
print(summary_s2)


=== Scenario 2: Multi-Class Curriculum Results ===
              top1_accuracy            macro_f1            \
                       mean       std      mean       std   
strategy                                                    
mcts_enhanced        0.0506  0.004384  0.027160  0.005484   
uniform              0.2297  0.001980  0.187105  0.013726   

              worst_10_class_recall       
                               mean  std  
strategy                                  
mcts_enhanced                   0.0  0.0  
uniform                         0.0  0.0  


## 5. Summary & Verification

All Tier-1 multi-seed experimental outputs have been saved to disk under `results/` for figure and table generation in Notebook 05 (`F6`).

In [6]:
print("Exported files in results/:")
for f in sorted(RESULTS_DIR.glob("tier1_*")):
    print(f"  {f.name} ({f.stat().st_size} bytes)")
print("Tier-1 MCTS-HNM benchmark execution complete.")

Exported files in results/:
  tier1_axis1_samplers.csv (3241 bytes)
  tier1_axis1_samplers.json (10212 bytes)
  tier1_axis2_tree_ablation.csv (1948 bytes)
  tier1_axis2_tree_ablation.json (5597 bytes)
  tier1_scenario2_curriculum.csv (541 bytes)
  tier1_scenario2_curriculum.json (2044 bytes)
Tier-1 MCTS-HNM benchmark execution complete.
